# 3. Analizzare il documento

Usa l'analyzer dell'esercizio 2 per analizzare il PDF indicato in `.env`.
`begin_analyze_binary` invia il file; `result()` attende il risultato.

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential

load_dotenv(".env", override=True)

import json
from pathlib import Path

documento = Path(os.environ["DOCUMENT_PATH"])
with ContentUnderstandingClient(
    endpoint=os.environ["AZURE_CONTENTUNDERSTANDING_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["AZURE_CONTENTUNDERSTANDING_API_KEY"]),
) as client:
    result = client.begin_analyze_binary(
        analyzer_id=os.environ["ANALYZER_ID"],
        binary_input=documento.read_bytes(),
    ).result()

## Leggi il risultato

Osserva testo, campi, confidence e source. Nel conto di Mario Rossi,
addebiti e accrediti sono entrambi 1587.10, mentre il saldo e 0.00.
La seconda pagina non deve raddoppiare gli importi.

In [ ]:
for content in result.contents:
    print(content.markdown)
    print(json.dumps(
        {name: field.as_dict() for name, field in (content.fields or {}).items()},
        ensure_ascii=False, indent=2,
    ))

Path("analisi.json").write_text(
    json.dumps(result.as_dict(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)